In [0]:
# Define Parameters
dbutils.widgets.text("catalog", "dbr_dev", "1. Unity Catalog Name")
dbutils.widgets.text("schema", "valeriimatviiv_bronze", "2. Target Schema Name")
dbutils.widgets.text("volume", "market_radar_landing", "3. Landing Volume Name")
dbutils.widgets.text("secret_scope", "valerii-matviiv-scope", "4. Secret Scope Name")
dbutils.widgets.text("secret_key", "finnhub-api-key", "5. Finnhub API Key Name")

# Retrieve Parameter Values
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")
secret_scope = dbutils.widgets.get("secret_scope")
secret_key = dbutils.widgets.get("secret_key")

# Verify Secret Scope Connection
try:
    api_key = dbutils.secrets.get(scope=secret_scope, key=secret_key)
    print(f"Successfully connected to secret scope: '{secret_scope}'")
except Exception as e:
    raise RuntimeError(f"Failed to fetch secret '{secret_key}' from scope '{secret_scope}': {e}")

# Create Schema and Volume under Unity Catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")

print(f"Target catalog & schema: {catalog}.{schema}")
print(f"Target volume: {catalog}.{schema}.{volume}")

# Establish Base Directories for Data Ingestion
base_volume_path = f"/Volumes/{catalog}/{schema}/{volume}"

landing_news_path = f"{base_volume_path}/landing/finnhub_news"
autoloader_checkpoint = f"{base_volume_path}/_state/checkpoints/finnhub_news"
autoloader_schema = f"{base_volume_path}/_state/schemas/finnhub_news"

# Create directories on Volume
dbutils.fs.mkdirs(landing_news_path)
dbutils.fs.mkdirs(autoloader_checkpoint)
dbutils.fs.mkdirs(autoloader_schema)

print(f"Landing path ready: {landing_news_path}")
print(f"Checkpoint path ready: {autoloader_checkpoint}")
print(f"Schema path ready: {autoloader_schema}")